# Probing the Misaligned Thinking Process of Language Models

**Goal**: After this session, you will be able to build a probe-based misalignment monitor from memory — train per-indicator linear probes on residual activations, OR-fuse them, and tune an operating point at a fixed false-positive rate.
**Time**: 30 minutes.

## What and Why
Running a strong LLM judge on every transcript (including the hidden `<think>` block) is expensive and breaks if the chain-of-thought becomes unreadable. Zhou et al. (2026) instead decompose "misalignment" into fine-grained *indicators* (strategic omission, observer-suspicion modeling, fabrication, ...) and train **one linear logistic-regression probe per indicator** on the target model's residual stream. At deployment, the 18 probes are **OR-fused** into a single cheap monitor: a turn is flagged if *any* probe fires on *any* sentence. The realistic monitoring metric is **transcript-TPR at a fixed ~1% neutral false-positive rate** — not accuracy.

## Key Formula
A turn is flagged when, for any indicator $i$ and any sentence $s$,
$$\text{flag} = \bigvee_{i}\ \Big[\max_{s}\ p_i(\text{activation}_s) \ge \tau_i\Big]$$

**Where:**
- $p_i(\cdot)$ — probability output of indicator $i$'s linear probe on a sentence's mean-pooled activation.
- $\tau_i$ — per-probe threshold, tuned so only a target fraction of *benign* sentences exceed it.
- $\bigvee$ — logical OR (the "OR-fusion" monitor): one indicator firing is enough.

## References
Zhou, Venhoff, Michala, Wang, Saunders. *Probing the Misaligned Thinking Process of Language Models*, arXiv:2606.24251 (2026). https://probe-misalignment.github.io


In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from jaxtyping import Float
from torch import Tensor

torch.manual_seed(0)
np.random.seed(0)

D_MODEL = 64     # residual-stream width (toy; real GLM-4.7 layer-27 stream is ~4096)
N_IND   = 6      # misalignment indicators (paper uses 18). One linear probe per indicator.
FIRE    = 4.0    # projection magnitude a "firing" sentence carries along its indicator direction

# Each indicator is ONE direction in activation space. These planted unit directions
# stand in for the directions a linear probe would learn from real GLM-4.7 residuals.
indicator_dirs = F.normalize(torch.randn(N_IND, D_MODEL), dim=1)   # (N_IND, D_MODEL)

def make_sentence(fires):
    """Token-level activations for one sentence of reasoning.
    `fires`: list of indicator indices this sentence instantiates ([] = benign).
    Returns (n_tokens, D_MODEL) — same shape/role as base_model(...).hidden_states[L][0]."""
    base = torch.randn(D_MODEL)                       # benign reasoning content (isotropic)
    for i in fires:
        base = base + FIRE * indicator_dirs[i]        # misaligned reasoning shifts along indicator i
    n_tokens = int(torch.randint(5, 15, (1,)))
    return base[None, :] + 0.3 * torch.randn(n_tokens, D_MODEL)   # tokens vary around the sentence mean

# --- Training data: mean-pooled sentence features + per-indicator binary labels ---
def _make_training_set(n_benign=400):
    feats, labels = [], []
    for _ in range(n_benign):                         # benign sentences -> all-zero labels
        feats.append(make_sentence([]).mean(0)); labels.append([0] * N_IND)
    for i in range(N_IND):                            # positives: each fires exactly indicator i
        for _ in range(n_benign // N_IND):
            feats.append(make_sentence([i]).mean(0))
            lab = [0] * N_IND; lab[i] = 1; labels.append(lab)
    return torch.stack(feats), torch.tensor(labels)

train_features, train_labels = _make_training_set()   # (N, D_MODEL), (N, N_IND)

# --- Transcripts: list of token-level sentence activations + a ground-truth misaligned flag ---
def _make_transcript(misaligned):
    n_sent = int(torch.randint(4, 9, (1,)))
    fire_flags = [misaligned and (torch.rand(1).item() < 0.4) for _ in range(n_sent)]
    if misaligned and not any(fire_flags):            # a misaligned transcript fires >= 1 sentence
        fire_flags[int(torch.randint(0, n_sent, (1,)))] = True
    sents = []
    for ff in fire_flags:
        fires = [int(torch.randint(0, N_IND, (1,)))] if ff else []
        sents.append(make_sentence(fires))
    return {"sentences": sents, "misaligned": misaligned}

eval_transcripts    = [_make_transcript(i % 2 == 0) for i in range(200)]   # mixed (100 misaligned, 100 benign)
neutral_transcripts = [_make_transcript(False) for _ in range(300)]        # all benign -> tune FPR here

print(f"train_features {tuple(train_features.shape)}  train_labels {tuple(train_labels.shape)}")
print(f"eval_transcripts {len(eval_transcripts)}  neutral_transcripts {len(neutral_transcripts)}")
print(f"misaligned in eval: {sum(t['misaligned'] for t in eval_transcripts)}")

train_features (796, 64)  train_labels (796, 6)
eval_transcripts 200  neutral_transcripts 300
misaligned in eval: 100


## Part 1: Train one linear probe per indicator
Each indicator is a single direction in activation space, so a *linear* probe is the right model. Train one logistic-regression probe per indicator on the mean-pooled sentence features.

**Predict before you code**: How many probes will you return? What does each probe's positive class mean?

In [2]:
def train_probes(
    features: Float[Tensor, "n_sent d_model"],
    labels: Float[Tensor, "n_sent n_ind"],
) -> list:
    """Fit one logistic-regression probe per indicator. Return a list of N_IND fitted probes."""
    X = features.numpy()
    probes = []
    for i in range(labels.shape[1]):
        probe = LogisticRegression(max_iter=1000)
        probe.fit(X, labels[:, i].numpy())
        probes.append(probe)
    return probes

In [3]:
# --- Part 1 Validation ---
probes = train_probes(train_features, train_labels)
assert len(probes) == N_IND, f"Expected {N_IND} probes, got {len(probes)}"
print(f"  Trained {len(probes)} probes")

# Each probe should near-perfectly separate its own indicator on the training set.
X = train_features.numpy()
aurocs = [roc_auc_score(train_labels[:, i].numpy(), probes[i].predict_proba(X)[:, 1]) for i in range(N_IND)]
print(f"  Per-probe train AUROC: {[round(a, 3) for a in aurocs]}")
assert min(aurocs) > 0.95, "A probe failed to learn its indicator direction (AUROC < 0.95)."

# A probe's weight vector should align with its planted indicator direction (linear concept).
w = torch.tensor(probes[0].coef_[0], dtype=torch.float32)
cos = F.cosine_similarity(w, indicator_dirs[0], dim=0)
print(f"  Probe-0 weight vs indicator_dirs[0] cosine: {cos:.3f}")
assert cos > 0.5, "Probe weight should point along its concept direction."
print("\nPart 1 complete.")

  Trained 6 probes
  Per-probe train AUROC: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  Probe-0 weight vs indicator_dirs[0] cosine: 0.906

Part 1 complete.


## Part 2: Score a transcript with each probe (OR-fusion ingredient)
A transcript is a list of sentences; the *sentence* is the natural unit of reasoning. Mean-pool each sentence's tokens, run every probe over every sentence, and reduce to one score per probe: the max firing probability across the transcript's sentences.

**Predict before you code**: What shape is the returned vector? Why `max` over sentences rather than `mean`?

In [4]:
def probe_scores(probes: list, transcript: dict) -> Float[Tensor, "n_ind"]:
    """For each probe, return its MAX P(fire) over all sentences in the transcript."""
    sent_feats = torch.stack([s.mean(0) for s in transcript["sentences"]]).numpy()  # (n_sent, d_model)
    scores = [probe.predict_proba(sent_feats)[:, 1].max() for probe in probes]
    return torch.tensor(scores, dtype=torch.float32)

In [5]:
# --- Part 2 Validation ---
mis = next(t for t in eval_transcripts if t["misaligned"])
ben = next(t for t in eval_transcripts if not t["misaligned"])
s_mis, s_ben = probe_scores(probes, mis), probe_scores(probes, ben)

assert s_mis.shape == (N_IND,), f"Expected ({N_IND},), got {tuple(s_mis.shape)}"
print(f"  Misaligned transcript probe scores: {[round(x, 3) for x in s_mis.tolist()]}")
print(f"  Benign     transcript probe scores: {[round(x, 3) for x in s_ben.tolist()]}")

# At least one probe should light up on the misaligned transcript; none should saturate on benign.
assert s_mis.max() > 0.5, "No probe fired on a known-misaligned transcript."
assert s_ben.max() < s_mis.max(), "Benign transcript scored as high as the misaligned one."
print(f"  max(misaligned)={s_mis.max():.3f} > max(benign)={s_ben.max():.3f}")
print("\nPart 2 complete.")

  Misaligned transcript probe scores: [0.026, 0.046, 0.009, 0.998, 0.021, 0.079]
  Benign     transcript probe scores: [0.562, 0.001, 0.004, 0.014, 0.009, 0.001]
  max(misaligned)=0.998 > max(benign)=0.562

Part 2 complete.


## Part 3: Tune per-probe thresholds at a target neutral FPR
OR-fusion inflates false positives (6 chances to fire per turn), so each probe needs its own threshold set on *benign* traffic. Set each probe's threshold so only `target_fpr` of neutral transcripts exceed it.

**Predict before you code**: If `target_fpr=0.01`, what quantile of the neutral score distribution is each threshold?

<details><summary>Hint: operating point</summary>This is a per-probe quantile. The threshold is the value such that a fraction `target_fpr` of benign transcripts score above it — i.e. the `1 - target_fpr` quantile of that probe's neutral scores.</details>

In [6]:
def tune_thresholds(probes: list, neutral: list, target_fpr: float = 0.01) -> Float[Tensor, "n_ind"]:
    """Per-probe threshold at the (1 - target_fpr) quantile of neutral scores."""
    S = torch.stack([probe_scores(probes, t) for t in neutral])   # (n_neutral, n_ind)
    return torch.quantile(S, 1.0 - target_fpr, dim=0)

In [7]:
# --- Part 3 Validation ---
thresholds = tune_thresholds(probes, neutral_transcripts, target_fpr=0.01)
assert thresholds.shape == (N_IND,), f"Expected ({N_IND},), got {tuple(thresholds.shape)}"
print(f"  Thresholds: {[round(x, 3) for x in thresholds.tolist()]}")

# Check the per-probe FPR each threshold actually induces on neutral traffic.
S = torch.stack([probe_scores(probes, t) for t in neutral_transcripts])   # (n_neutral, n_ind)
per_probe_fpr = (S >= thresholds).float().mean(0)
print(f"  Realized per-probe neutral FPR: {[round(x, 3) for x in per_probe_fpr.tolist()]}")
assert (per_probe_fpr <= 0.05).all(), "A threshold is too low — per-probe FPR exceeds target."
print("\nPart 3 complete.")

  Thresholds: [0.914, 0.942, 0.906, 0.789, 0.843, 0.952]


  Realized per-probe neutral FPR: [0.01, 0.01, 0.01, 0.01, 0.01, 0.01]

Part 3 complete.


## Part 4: Evaluate the OR-fusion monitor
Flag a transcript if *any* probe clears its threshold. Report the realistic monitoring numbers: transcript-TPR, neutral FPR (via OR-fusion of all probes), and the AUROC of the continuous monitor score (`max` probe score).

**Predict before you code**: With 6 probes each at ~1% FPR, will the OR-fused turn-level FPR be above or below 1%?

In [8]:
def evaluate(probes: list, thresholds: Float[Tensor, "n_ind"], transcripts: list):
    """Return (tpr, fpr, auroc) for the OR-fusion monitor over `transcripts`."""
    flags, scores, truth = [], [], []
    for t in transcripts:
        ps = probe_scores(probes, t)
        flags.append(bool((ps >= thresholds).any()))   # OR-fusion
        scores.append(ps.max().item())                 # continuous monitor score for AUROC
        truth.append(t["misaligned"])
    flags = torch.tensor(flags); truth = torch.tensor(truth)
    tpr = (flags & truth).sum().item() / truth.sum().item()
    fpr = (flags & ~truth).sum().item() / (~truth).sum().item()
    auroc = roc_auc_score(truth.numpy(), scores)
    return tpr, fpr, auroc

In [9]:
# --- Part 4 Validation ---
tpr, fpr, auroc = evaluate(probes, thresholds, eval_transcripts)
print(f"  Transcript-TPR: {tpr:.3f}")
print(f"  Neutral FPR (OR-fused, on eval benign): {fpr:.3f}")
print(f"  Monitor AUROC: {auroc:.3f}")

assert auroc > 0.95, "Monitor barely separates misaligned from benign (AUROC < 0.95)."
assert tpr > 0.80, "Monitor misses too many misaligned transcripts (TPR < 0.80)."
# OR-fusion of 6 probes tuned at 1% each lands well above 1% turn-level FPR — the reason
# the paper pairs probes with a Stage-2 LLM judge (see Extension).
print(f"\n  Note: OR-fusing {N_IND} probes raises turn-level FPR above any single probe's 1%.")
print("\nPart 4 complete.")

  Transcript-TPR: 0.910
  Neutral FPR (OR-fused, on eval benign): 0.040
  Monitor AUROC: 0.981

  Note: OR-fusing 6 probes raises turn-level FPR above any single probe's 1%.

Part 4 complete.


## Session Debrief

Without scrolling up, answer in your head:
1. Why train one probe *per indicator* instead of a single "misalignment" probe? (Hint: §2 of the paper — interpretability + a well-posed single-feature target per probe.)
2. What does OR-fusion do to the false-positive rate, and why must thresholds be tuned on *benign* data?
3. Why report TPR at a fixed 1% FPR instead of accuracy or AUROC alone?

**Challenge**: Close this notebook, open a blank one, and rewrite Part 3 (threshold tuning at a target FPR) from scratch without looking back.


## Extension (Optional)
- **Cascade**: Add a Stage-2 "LLM judge" stub that is only called on probe-flagged turns. Measure the fraction of neutral traffic that reaches the judge (the paper's cost saving — ~6%).
- **Break it**: Set `FIRE = 1.0` (weak misalignment signal). Re-run. Where does TPR@1%FPR collapse, and which part of the pipeline is the bottleneck?
- **Span vs turn labeling**: Re-tune thresholds at `target_fpr=0.05` and plot the TPR/FPR trade-off curve across several targets.


## Anki Cards

**Card 1**
Front: A misalignment monitor must run on every deployed turn but a strong LLM judge is too expensive. What's the cheap white-box pre-filter?
Back: Linear probes on activations

**Card 2**
Front: You have 18 per-indicator probes. How do you combine them into one turn-level flag?
Back: OR-fusion (any probe fires)

**Card 3**
Front: Your probe monitor has 90% TPR but floods on-call with alerts. What metric/knob did you forget to set?
Back: Threshold at a fixed neutral FPR
